# WeatherNext 3 → FAO-56 ET₀ (Colab)

Downloads the five FAO-56 input fields plus precipitation, adds DEM elevation, computes
**hourly** reference evapotranspiration (FAO-56 Eq. 53) and aggregates to daily totals.
Output: an hourly and a daily NetCDF, each carrying precipitation, temperature and ET₀.

Cell 4 runs `--self-test` first, which reproduces FAO-56 Example 19 without touching
Earth Engine — if that passes, the physics chain is intact.

In [ ]:
#@title 1. Get the code and install dependencies
!git clone -q https://github.com/majidniazkar/weathernext3-forecast.git
%cd weathernext3-forecast
!pip install -q -r requirements.txt

In [ ]:
#@title 2. Authenticate Earth Engine
import ee

EE_PROJECT = "my-ee-project"  #@param {type:"string"}
ee.Authenticate()
ee.Initialize(project=EE_PROJECT)
print("Earth Engine ready")

In [ ]:
#@title 3. Settings
WEST, SOUTH, EAST, NORTH = 0.0, 0.0, 0.0, 0.0 # set your bounding box, degrees
DAYS        = 14          #@param {type:"number"}
STAT        = "mean"      #@param ["mean", "p10", "p25", "p50", "p75", "p90"]
PRESSURE    = "elevation" #@param ["elevation", "mslp"]
UTC_OFFSET  = 0           #@param {type:"number"}   # 1 = CET, for daily aggregation

In [ ]:
#@title 4. Check the FAO-56 implementation, then download and compute
!python weathernext3_eto.py --self-test
!python weathernext3_eto.py --project "$EE_PROJECT" --bbox $WEST $SOUTH $EAST $NORTH --days $DAYS --stat $STAT --pressure-source $PRESSURE --utc-offset $UTC_OFFSET

In [ ]:
#@title 5. Inspect and download
import glob

import matplotlib.pyplot as plt
import xarray as xr
from google.colab import files

daily = xr.open_dataset(sorted(glob.glob("weathernext3_eto_*_daily.nc"))[-1])
hourly = xr.open_dataset(sorted(glob.glob("weathernext3_eto_*_hourly.nc"))[-1])
print(dict(daily.sizes), list(daily.data_vars))

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
daily["eto"].sum("time").plot(ax=axes[0], cmap="YlOrBr")
axes[0].set_title("Total ET0 over the forecast (mm)")
daily["precipitation"].sum("time").plot(ax=axes[1], cmap="BuPu")
axes[1].set_title("Total precipitation (mm)")
area = daily.mean(("latitude", "longitude"))
area["eto"].plot(ax=axes[2], marker="o", label="ET0")
area["precipitation"].plot(ax=axes[2], marker="s", label="precipitation")
axes[2].set_ylabel("mm/day"); axes[2].legend(); axes[2].set_title("Area mean")
fig.tight_layout()

for f in sorted(glob.glob("weathernext3_eto_*.nc")):
    files.download(f)

The hourly file also carries `vpd`, `wind_speed_2m`, `net_radiation`, `solar_radiation`,
`pressure` and `elevation`, so each ET₀ term can be inspected. `hours_in_day` in the daily file
flags partial days at the start and end of the run — drop those rows before using daily totals.

To aggregate ET₀, precipitation and every other variable per sub-basin, feed the hourly or daily
file to [`WeatherNext3_SubBasin_Aggregation_Colab.ipynb`](WeatherNext3_SubBasin_Aggregation_Colab.ipynb)
with a shapefile of your basins — it writes one Excel workbook per NetCDF, one sheet per variable.